# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library. The dataset contains survey data on mental health indicators among residents, including demographic details and standardized psychological assessments (GAD-7, PHQ-9, PSQ).

### Dataset Source
The dataset is defined via a Croissant schema. The schema is public and retrieves all required relationships, record sets, fields, and columns by their uniquely identifying `@id` field.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata using `mlcroissant` to ensure access to all referenced objects and schemas.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview

Explore the available record sets and their structure. All `mlcroissant` entities are referenced by their `@id` as defined in the Croissant schema. Let's first list the available record sets and their fields.


In [ ]:
# List all available record sets by their @id
record_sets = list(dataset.record_sets)
print("Available record sets (by @id):\n-----------------------------")
for rs in record_sets:
    print(f"@id: {rs.id} | Name: {rs.name} | Description: {rs.description}")

# For the main survey, the likely record set id is 'cr:RecordSet', but always check actual IDs
# List the fields for each record set
for rs in record_sets:
    print(f"\nFields for record set {rs.id}:")
    for field in rs.fields:
        print(f" - Field @id: {field.id}\tName: {field.name}\tType: {field.data_type}")

## 3. Data Extraction

Let's select the main survey record set and load its contents into a DataFrame for analysis. Entities are referenced only by their `@id`.

> **Tip:** To see all data, loop over all record set ids; below we load them all as a dictionary of DataFrames by record set `@id`.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs.id for rs in dataset.record_sets]

# Load all record sets into dataframes by @id
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set: {record_set_id}")
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")

# If only one main dataset, show its columns
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id and main_rs_id in dataframes:
    print(f"\nColumns in main record set (@id={main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's perform basic data filtering, normalization, and grouping by key attributes. All columns are referenced by their `@id` or field names as per the schema and extraction above. If you want to reference specific columns (`@id`), consult the printed columns above.

In [ ]:
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Choose fields/columns by @id or schema fieldname
# For example, let's assume GAD-7 total score is '@id': 'gad7_total_score'
# You can adapt the field name below to match the dataset's actual columns

main_rs_id = main_rs_id  # previously set

df_main = dataframes[main_rs_id]

# Try to auto-detect a numeric mental health score field
possible_numeric_fields = [col for col in df_main.columns if any(x in col.lower() for x in ['total', 'score', 'phq', 'gad', 'psq'])]
numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df_main.columns[0]
print(f"Using numeric field: {numeric_field}")

# Set an arbitrary threshold (e.g., 10 for clinical cutoffs)
threshold = 10
filtered_df = df_main[df_main[numeric_field].astype(float) > threshold]
print(f"\nRecords with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
) / filtered_df[numeric_field].astype(float).std()
print(f"\nNormalized {numeric_field}:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a logical category (e.g., 'village', 'level_of_education', etc.)
# Try to auto-detect one
group_fields = [col for col in df_main.columns if any(x in col.lower() for x in ['village', 'education', 'gender'])]
if group_fields:
    group_field = group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    display(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship to a demographic group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the normalized score
plt.figure(figsize=(7, 4))
sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True, bins=20)
plt.title(f"Normalized Distribution of {numeric_field}")
plt.xlabel(f"{numeric_field} (normalized)")
plt.ylabel("Count")
plt.show()

# Boxplot by group if grouping field exists
if group_fields:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field].astype(float))
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, referenced all data elements by their `@id`, inspected the structure and metadata, performed basic analysis on numeric mental health assessment scores, and visualized findings by demographic variables. This workflow demonstrates how to use the `mlcroissant` library to rapidly onboard and analyze FAIR data, leveraging Croissant schema structure for reproducible workflows.

Further analyses may include investigating correlations between different assessments and demographics, handling missing values, and building predictive models.
